# HW3: Convolutional Neural Networks from Scratch
## Modern Computer Vision, Technion (Spring 2026)

In this assignment, you will implement **Convolutional Neural Networks (CNNs)** from the ground up using only NumPy-style operations and PyTorch's functional unfold/fold utilities.

### Learning Goals
- Understand how convolution works under the hood
- Implement efficient convolution using unfold/fold
- Implement backpropagation through conv and pooling layers
- Build a complete CNN training pipeline
- Train on CIFAR-10 and compare with PyTorch's implementation

### What you've already built (HW2)
- Linear layers (forward + backward)
- ReLU activation
- Cross-entropy loss
- SGD optimizer
- MLP training pipeline

### What you're building today
- **conv2d**: 2D convolution operation (the core operation of CNNs)
- **max_pool2d**: Max pooling for dimension reduction
- **MomentumSGD**: SGD with momentum for better optimization
- **ConvNet**: A complete CNN for CIFAR-10
- **Training pipeline**: Train and evaluate on CIFAR-10

## Submission Requirements

**IMPORTANT:** Submit this notebook **with all outputs** — run all cells before submitting. Notebooks without outputs will lose points. Make sure your `STUDENT_ID` is set correctly.

**Student ID:**

In [ ]:
# Fill in your student ID (as a string), e.g. STUDENT_ID = "123456789"
STUDENT_ID = ""

## Setup & Imports

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time
from collections import defaultdict

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
# Use GPU if available. The from-scratch layers are device-agnostic
# (every allocation follows the input's device), so the same code runs
# on CPU or CUDA -- only this line and the .to(device) calls change.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## Part 0: Provided Code from HW2

These implementations from HW2 are provided so the notebook is self-contained.

### Context Tape Autograd Pattern

We use a context-tape pattern for automatic differentiation:
- Each operation appends `[backward_fn, args]` to a list (ctx)
- `backward()` pops operations in reverse order and applies backward functions
- This implements reverse-mode autodiff (backpropagation)

In [ ]:
def ensure_grad(tensors):
    """Create .grad (zeros) for each tensor that doesn't have one yet.

    Same as HW2, with one addition: non-tensor tape entries (padding, stride, 
    dilation ints carried by conv backward) are skipped via the is_tensor check.
    """
    for t in tensors:
        if torch.is_tensor(t) and (not hasattr(t, 'grad') or t.grad is None):
            t.grad = torch.zeros_like(t)


def backward(loss, ctx):
    """
    Backward pass over the context tape.

    Seed loss.grad = 1, then pop operations in reverse order and replay them.
    Each tape entry is [backward_fn, [output, *inputs]]: the backward_fn reads
    output.grad (the gradient flowing in) and accumulates into each input's
    .grad (the gradient flowing out).
    """
    loss.grad = torch.ones_like(loss)
    while ctx:
        backward_fn, args = ctx.pop()
        ensure_grad(args)
        backward_fn(*args)


class Module:
    """Base class for all neural network modules.

    Parameters are plain tensors, registered by attribute name in
    self._parameters. Sub-modules are registered by attribute name in
    self._modules. parameters() collects both, recursively.
    """

    def __init__(self):
        self._parameters = []  # names of parameter-tensor attributes
        self._modules = []     # names of sub-module attributes

    def to(self, device):
        """Move all parameters to a device."""
        for name in self._parameters:
            setattr(self, name, getattr(self, name).to(device))
        for name in self._modules:
            getattr(self, name).to(device)
        return self

    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)

    def parameters(self):
        """Collect all learnable parameters (recursively)."""
        params = []
        for name in self._parameters:
            params.append(getattr(self, name))
        for name in self._modules:
            params.extend(getattr(self, name).parameters())
        return params

    def train(self):
        """Set module to training mode (no-op in the from-scratch version)."""
        return self

    def eval(self):
        """Set module to evaluation mode (no-op in the from-scratch version)."""
        return self

    def zero_grad(self):
        """Zero out gradients for all parameters."""
        for param in self.parameters():
            param.grad = None

### ReLU Activation

In [ ]:
def relu_forward(x):
    """Forward pass of ReLU: y = max(x, 0)."""
    return torch.clamp(x, min=0)


def relu_backward(y, x):
    """Backward pass of ReLU: gradient flows through where x > 0."""
    x.grad += y.grad * (x > 0).float()


def relu(x, ctx=None):
    """ReLU with autograd support: records [relu_backward, [y, x]] onto ctx."""
    y = relu_forward(x)
    if ctx is not None:
        ctx.append([relu_backward, [y, x]])
    return y


class ReLU(Module):
    """ReLU activation layer."""

    def forward(self, x, ctx=None):
        return relu(x, ctx=ctx)

### Cross-Entropy Loss

In [ ]:
def cross_entropy_forward(logits, targets):
    """
    Forward pass of cross-entropy loss.
    logits: shape (N, C); targets: shape (N,) with class indices.
    Returns: scalar loss.
    """
    N = logits.shape[0]
    # Numerical stability: subtract max
    logits_stable = logits - logits.max(dim=1, keepdim=True)[0]
    log_softmax = logits_stable - torch.log(torch.exp(logits_stable).sum(dim=1, keepdim=True))
    loss = -log_softmax[torch.arange(N), targets].mean()
    return loss


def cross_entropy_loss_backward(loss, logits, targets):
    """
    Backward pass of cross-entropy loss (reads loss.grad).
    Accumulates (softmax(logits) - one_hot(targets)) / N into logits.grad,
    scaled by the incoming loss.grad.
    """
    N = logits.shape[0]
    logits_stable = logits - logits.max(dim=1, keepdim=True)[0]
    probs = torch.exp(logits_stable) / torch.exp(logits_stable).sum(dim=1, keepdim=True)
    grad = probs.clone()
    grad[torch.arange(N), targets] -= 1
    grad /= N
    logits.grad += loss.grad * grad


def cross_entropy_loss(logits, targets, ctx=None):
    """Cross-entropy loss with autograd support."""
    loss = cross_entropy_forward(logits, targets)
    if ctx is not None:
        ctx.append([cross_entropy_loss_backward, [loss, logits, targets]])
    return loss

### Linear Layer

In [ ]:
def linear_forward(x, w, b=None):
    """
    Forward pass of linear layer: y = x @ w.T + b
    x: (N, D_in); w: (D_out, D_in); b: (D_out,)
    Returns: y of shape (N, D_out)
    """
    y = x @ w.t()
    if b is not None:
        y = y + b
    return y


def linear_backward(y, x, w, b):
    """Backward pass of linear layer (reads y.grad)."""
    x.grad += y.grad @ w
    w.grad += y.grad.t() @ x
    if b is not None:
        b.grad += y.grad.sum(dim=0)


def linear(x, w, b=None, ctx=None):
    """Linear with autograd support: records [linear_backward, [y, x, w, b]]."""
    y = linear_forward(x, w, b)
    if ctx is not None:
        ctx.append([linear_backward, [y, x, w, b]])
    return y


class Linear(Module):
    """Linear (fully connected) layer."""

    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        # He initialization (plain tensors, no requires_grad needed)
        self.weight = torch.randn(out_features, in_features) * np.sqrt(2.0 / in_features)
        self._parameters = ["weight"]

        if bias:
            self.bias = torch.zeros(out_features)
            self._parameters.append("bias")
        else:
            self.bias = None

    def forward(self, x, ctx=None):
        return linear(x, self.weight, self.bias, ctx=ctx)

### SGD Optimizer

In [ ]:
class SGD:
    """Stochastic Gradient Descent optimizer."""
    
    def __init__(self, params, lr=0.01):
        self.params = list(params)
        self.lr = lr
    
    def step(self):
        """Update parameters using their gradients."""
        for param in self.params:
            if param.grad is not None:
                param.data -= self.lr * param.grad
    
    def zero_grad(self):
        """Zero out gradients."""
        for param in self.params:
            param.grad = None

### Training & Evaluation Utilities

In [ ]:
def accuracy(logits, targets):
    """Compute classification accuracy.
    logits: shape (N, C); targets: shape (N,)
    """
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def evaluate(model, dataloader, loss_fn, device='cpu'):
    """Evaluate model on a dataset. Returns: (loss, accuracy).

    No ctx is passed, so nothing is recorded for backward.
    """
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    model.eval()
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        logits = model.forward(x)          # no ctx -> no recording
        loss = loss_fn(logits, y)

        total_loss += loss.item() * x.shape[0]
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_samples += x.shape[0]

    return total_loss / total_samples, total_correct / total_samples


def train_epoch(model, train_loader, optimizer, loss_fn, device='cpu'):
    """Train for one epoch. Returns: (average_loss, accuracy)."""
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        ctx = []                              # fresh backward tape per batch

        # Forward pass (records operations onto ctx)
        logits = model.forward(x, ctx=ctx)
        loss = loss_fn(logits, y, ctx=ctx)

        # Backward pass
        optimizer.zero_grad()
        backward(loss, ctx)

        # Update
        optimizer.step()

        # Statistics
        total_loss += loss.item() * x.shape[0]
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_samples += x.shape[0]

    return total_loss / total_samples, total_correct / total_samples

---
# Part 1: Convolution (conv2d)

## Overview

Convolution is the core operation of CNNs. A convolution applies a small filter (kernel) across an input image, sliding it over every position and computing element-wise products.

### High-level idea

For an image `x` of shape `(N, C_in, H, W)` and kernel `w` of shape `(C_out, C_in, kH, kW)`:

1. **Extract patches**: Use `F.unfold` to extract all kernel-sized patches from the input
   - Output shape: `(N, C_in * kH * kW, L)` where L is the number of patches

2. **Reshape kernel**: Flatten the kernel from `(C_out, C_in, kH, kW)` to `(C_out, C_in * kH * kW)`

3. **Matrix multiply**: Each patch is a column in the unfolded input. Multiply kernel with these patches:
   - `y_flat = w_flat @ x_unfold` → shape `(N, C_out, L)`

4. **Reshape output**: Reshape `y_flat` back to `(N, C_out, H_out, W_out)`

5. **Add bias**: Add bias term to the output

### Output spatial dimensions

```
H_out = floor((H + 2*padding - dilation*(kH-1) - 1) / stride + 1)
W_out = floor((W + 2*padding - dilation*(kW-1) - 1) / stride + 1)
```

### Why unfold/fold?

- Converts convolution to a series of matrix multiplications
- Much faster than explicit loops
- Makes backpropagation straightforward: just compute gradients of matrix ops
- The gradients automatically scatter back to input positions via fold

## Forward Pass: Implementing conv2d_forward

Your task: Implement the forward pass of 2D convolution using `F.unfold` and `F.fold`.

**Steps:**
1. Use `F.unfold` to extract patches from the input
2. Reshape the kernel to 2D for matrix multiplication
3. Compute the matrix product
4. Reshape the output to the correct spatial dimensions
5. Add the bias if provided

**Hints:**
- `F.unfold` returns shape `(N, C*kH*kW, L)` where L = H_out * W_out
- You can compute H_out and W_out using the formula above
- `F.fold` takes `(N, C, L)` tensor and kernel_size and unfolds it back to `(N, C, H, W)`
- For bias, use broadcasting (add to shape `(N, C_out, 1, 1)` to broadcast across spatial dims)

In [ ]:
def conv2d_forward(x, w, b=None, padding=0, stride=1, dilation=1):
    """
    Forward pass of 2D convolution.
    
    Args:
        x: Input tensor of shape (N, C_in, H, W)
        w: Kernel weights of shape (C_out, C_in, kH, kW)
        b: Bias of shape (C_out,) or None
        padding: Zero padding amount (int or tuple)
        stride: Stride amount (int or tuple)
        dilation: Dilation amount (int or tuple)
    
    Returns:
        y: Output tensor of shape (N, C_out, H_out, W_out)
    """
    N, C_in, H, W = x.shape
    C_out, _, kH, kW = w.shape
    
    # TODO: Implement conv2d forward pass
    raise NotImplementedError("TODO: Implement conv2d forward pass using unfold/fold")

## Sanity Check: Forward Pass

Compare your implementation with PyTorch's `F.conv2d`.

In [ ]:
# Test conv2d forward
torch.manual_seed(42)
N, C_in, H, W = 2, 3, 8, 8
C_out, kH, kW = 4, 3, 3
padding, stride = 1, 1

x_test = torch.randn(N, C_in, H, W)
w_test = torch.randn(C_out, C_in, kH, kW)
b_test = torch.randn(C_out)

# Your implementation
y_yours = conv2d_forward(x_test, w_test, b_test, padding=padding, stride=stride)

# PyTorch reference
y_torch = F.conv2d(x_test, w_test, b_test, padding=padding, stride=stride)

print(f"Your output shape:    {tuple(y_yours.shape)}")
print(f"PyTorch output shape: {tuple(y_torch.shape)}")
print(f"Max difference: {(y_yours - y_torch).abs().max().item():.2e}")
assert torch.allclose(y_yours, y_torch, atol=1e-5), "Forward pass doesn't match PyTorch!"
print("✓ Forward pass matches PyTorch!")

## Backward Pass: Implementing conv2d_backward

The backward pass computes gradients w.r.t. input, weights, and bias.

**Key insight:** If forward is `y = (x_unfold @ w_flat.T) + b`, then:

- **Gradient w.r.t. w**: `w.grad = y.grad @ x_unfold.T` (after unfolding y.grad)
- **Gradient w.r.t. x**: Use `F.fold` on `(w.T @ y_grad_flat)` to scatter gradients back to input positions
- **Gradient w.r.t. b**: Sum y.grad across batch and spatial dimensions

Your task: Implement the backward pass.

**Hints:**
- You'll need to call `F.unfold` again on the input (you have it in the context)
- Use `F.fold` with `kernel_size=(kH, kW)` to "unpad" the gradients back to input size, or use padding/stride to account for the original convolution params
- Be careful about which tensors get the gradients (w.grad, x.grad, b.grad)

In [ ]:
def conv2d_backward(y, x, w, b, padding, stride, dilation):
    """
    Backward pass of 2D convolution.

    Reads the upstream gradient from y.grad and accumulates gradients into
    x.grad, w.grad, and (if present) b.grad.

    Args:
        y: Output of the forward pass, shape (N, C_out, H_out, W_out); grad in y.grad
        x: Input tensor, shape (N, C_in, H, W)
        w: Kernel weights, shape (C_out, C_in, kH, kW)
        b: Bias, shape (C_out,) or None
        padding, stride, dilation: Conv parameters (tuples)
    """
    # TODO: Implement conv2d backward pass
    raise NotImplementedError("TODO: Implement conv2d backward pass")

## Sanity Check: Backward Pass

Compare your gradients with PyTorch's autograd.

In [ ]:
# Test conv2d backward
torch.manual_seed(42)
N, C_in, H, W = 2, 3, 8, 8
C_out, kH, kW = 4, 3, 3
padding, stride = 1, 1

x_test = torch.randn(N, C_in, H, W, requires_grad=True)
w_test = torch.randn(C_out, C_in, kH, kW, requires_grad=True)
b_test = torch.randn(C_out, requires_grad=True)

# PyTorch reference (dummy loss = sum, so the upstream gradient is all ones)
y_torch = F.conv2d(x_test, w_test, b_test, padding=padding, stride=stride)
y_torch.sum().backward()

# Your implementation: run the forward, seed the same upstream gradient
# (ones, matching loss = y.sum()), then call your conv2d_backward.
x_ours, w_ours, b_ours = x_test.detach(), w_test.detach(), b_test.detach()
y_ours = conv2d_forward(x_ours, w_ours, b_ours, padding=padding, stride=stride)
y_ours.grad = torch.ones_like(y_ours)
ensure_grad([x_ours, w_ours, b_ours])
conv2d_backward(y_ours, x_ours, w_ours, b_ours,
                (padding, padding), (stride, stride), (1, 1))

for name, ours, ref in [("x", x_ours, x_test), ("w", w_ours, w_test), ("b", b_ours, b_test)]:
    diff = (ours.grad - ref.grad).abs().max().item()
    print(f"{name}.grad max diff: {diff:.2e}")
    assert torch.allclose(ours.grad, ref.grad, atol=1e-4), f"{name}.grad doesn't match PyTorch!"
print("✓ Backward pass matches PyTorch!")

## Conv2d Layer Wrapper

Wrap the conv2d forward and backward functions in a Module class.

### Your Task: Implement Conv2d Layer

Create a `Conv2d` module that wraps your `conv2d_forward` and `conv2d_backward` functions.

**Requirements:**
- Constructor: initialize weights and bias as plain tensors and register their names in `self._parameters`
- Weight initialization: use He initialization (scale by `sqrt(2 / (C_in * kH * kW))`)
- `forward` method: call `conv2d_forward`, and when `ctx` is given, record the backward op on the tape; return the output

In [ ]:
class Conv2d(Module):
    """2D Convolution layer."""

    def __init__(self, in_channels, out_channels, kernel_size, padding=0, stride=1, dilation=1, bias=True):
        """
        Args:
            in_channels: Number of input channels
            out_channels: Number of output channels
            kernel_size: Size of the convolutional kernel (int or tuple)
            padding: Padding amount (default: 0)
            stride: Stride amount (default: 1)
            dilation: Dilation amount (default: 1)
            bias: Whether to use bias (default: True)
        """
        super().__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.padding = padding if isinstance(padding, tuple) else (padding, padding)
        self.stride = stride if isinstance(stride, tuple) else (stride, stride)
        self.dilation = dilation if isinstance(dilation, tuple) else (dilation, dilation)

        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        self.kernel_size = kernel_size

        # TODO: initialize the weight (He init) and bias, and register their
        #       attribute names in self._parameters.
        raise NotImplementedError("TODO: Initialize Conv2d weights and bias")

    def forward(self, x, ctx=None):
        # TODO: call conv2d_forward, record the backward op on ctx, return the output.
        raise NotImplementedError("TODO: Implement Conv2d forward")

---
# Part 2: Max Pooling (max_pool2d)

## Overview

Max pooling reduces spatial dimensions by taking the maximum value in each non-overlapping (or overlapping with stride) kernel window.

### Key idea

1. Use `F.unfold` to extract patches (same as conv2d)
2. Take max along the channel dimension (the kernel values)
3. Also keep track of argmax indices (needed for backward)
4. Reshape back to spatial dimensions

### Backward

- Gradient only flows to the position that was the max
- Use the argmax indices to scatter gradients back
- Use `F.fold` to accumulate gradients to the input positions

In [ ]:
def max_pool2d_forward(x, kernel_size, padding=0, stride=None):
    """
    Forward pass of max pooling.
    
    Args:
        x: Input tensor of shape (N, C, H, W)
        kernel_size: Size of the pooling kernel (int or tuple)
        padding: Padding amount (default: 0)
        stride: Stride amount (default: kernel_size)
    
    Returns:
        y: Output tensor of shape (N, C, H_out, W_out)
        argmax_indices: Indices of max values (for backward pass)
    """
    if stride is None:
        stride = kernel_size
    
    if isinstance(kernel_size, int):
        kernel_size = (kernel_size, kernel_size)
    if isinstance(stride, int):
        stride = (stride, stride)
    
    N, C, H, W = x.shape
    kH, kW = kernel_size
    
    # TODO: Implement max_pool2d forward
    # Hint: use F.unfold, then max and argmax
    raise NotImplementedError("TODO: Implement max_pool2d forward")

In [ ]:
def max_pool2d_backward(y, x, kernel_size, padding, stride, argmax_indices):
    """
    Backward pass of max pooling.

    Reads the upstream gradient from y.grad and routes it back to the input
    positions that were the max (given by argmax_indices), accumulating into
    x.grad.

    Args:
        y: Output of the forward pass, shape (N, C, H_out, W_out); grad in y.grad
        x: Input tensor, shape (N, C, H, W)
        kernel_size, padding, stride: Same as forward
        argmax_indices: Indices of the max positions from the forward pass
    """
    # TODO: Implement max_pool2d backward
    raise NotImplementedError("TODO: Implement max_pool2d backward")

### MaxPool2d Layer

In [ ]:
class MaxPool2d(Module):
    """2D Max Pooling layer."""

    def __init__(self, kernel_size, padding=0, stride=None):
        """
        Args:
            kernel_size: Size of the pooling kernel
            padding: Padding amount (default: 0)
            stride: Stride amount (default: kernel_size)
        """
        super().__init__()
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size)
        self.kernel_size = kernel_size
        self.padding = padding
        self.stride = stride if stride is not None else kernel_size

    def forward(self, x, ctx=None):
        # TODO: call max_pool2d_forward, record the backward op on ctx (it also
        #       needs the argmax indices), and return the output.
        raise NotImplementedError("TODO: Implement MaxPool2d forward")

## Sanity Check: Max Pooling

In [ ]:
# Test max_pool2d forward
torch.manual_seed(42)
x_test = torch.randn(2, 3, 8, 8)

# Your implementation (max_pool2d_forward returns (output, argmax_indices))
y_yours, _ = max_pool2d_forward(x_test, kernel_size=2, stride=2)

# PyTorch reference
y_torch = F.max_pool2d(x_test, kernel_size=2, stride=2)

print(f"Your output shape:    {tuple(y_yours.shape)}")
print(f"PyTorch output shape: {tuple(y_torch.shape)}")
print(f"Max difference: {(y_yours - y_torch).abs().max().item():.2e}")
assert torch.allclose(y_yours, y_torch, atol=1e-5), "Max pooling doesn't match PyTorch!"
print("✓ Max pooling matches PyTorch!")

---
# Part 3: MomentumSGD Optimizer

## Overview

SGD with momentum accumulates gradients to smooth the optimization trajectory:

```
v_t = momentum * v_{t-1} + grad
param -= lr * v_t
```

This helps:
- Accelerate convergence in consistent directions
- Reduce oscillations in noisy gradients
- Escape shallow local minima

In [ ]:
class MomentumSGD:
    """SGD with momentum optimizer."""
    
    def __init__(self, params, lr=0.01, momentum=0.9):
        """
        Args:
            params: Iterable of parameters to optimize
            lr: Learning rate
            momentum: Momentum factor (default: 0.9)
        """
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum
        self.velocity = {id(param): torch.zeros_like(param) for param in self.params}
    
    def step(self):
        """Update parameters using momentum."""
        for param in self.params:
            if param.grad is not None:
                param_id = id(param)
                # TODO: Update velocity and parameter
                raise NotImplementedError("TODO: Implement MomentumSGD step")
    
    def zero_grad(self):
        """Zero out gradients."""
        for param in self.params:
            param.grad = None

## Flatten

Your `ConvNet` needs to turn the `(N, C, H, W)` feature maps from the conv stack into `(N, C*H*W)` vectors before the linear layers.

A bare `x.reshape(...)` would **break the backward tape**: the reshaped tensor is a new tensor with its own `.grad`, so gradients would never reach the conv layers. Implement `flatten` as a differentiable op that records itself on the tape, just like `relu` and `linear`.

In [ ]:
def flatten_backward(y, x):
    """Backward for flatten."""
    # TODO: accumulate the gradient from y.grad into x.grad (reshape it to x's shape)
    raise NotImplementedError("TODO: Implement flatten backward")


def flatten(x, ctx=None):
    """Forward: (N, C, H, W) -> (N, C*H*W). Record on the tape when ctx is given."""
    # TODO: compute y, append [flatten_backward, [y, x]] to ctx, and return y
    raise NotImplementedError("TODO: Implement flatten forward")

---
# Part 4: ConvNet Model for CIFAR-10

## Architecture Design

Design a CNN for CIFAR-10 classification. CIFAR-10 has:
- 32x32 RGB images (3 channels)
- 10 output classes

### Suggested Architecture

```
Conv2d(3, 32, kernel_size=3, padding=1)
  → ReLU
  → MaxPool2d(2, stride=2)  # 32x32 → 16x16
  
Conv2d(32, 64, kernel_size=3, padding=1)
  → ReLU
  → MaxPool2d(2, stride=2)  # 16x16 → 8x8
  
Conv2d(64, 128, kernel_size=3, padding=1)
  → ReLU
  → MaxPool2d(2, stride=2)  # 8x8 → 4x4
  
Flatten → Linear(128*4*4, 256) → ReLU → Linear(256, 10)
```

Feel free to adjust the architecture (more/fewer channels, different kernel sizes, etc.)

In [ ]:
class ConvNet(Module):
    """CNN for CIFAR-10 classification."""

    def __init__(self):
        super().__init__()
        # TODO: Build your architecture using Conv2d, ReLU, MaxPool2d, and Linear
        #       layers. Register every sub-layer's attribute name in self._modules
        #       so parameters() collects their weights. Final output: 10 classes.
        raise NotImplementedError("TODO: Design ConvNet architecture")

    def forward(self, x, ctx=None):
        # TODO: Implement the forward pass. Pass ctx to every layer so the backward
        #       tape is recorded, and flatten the feature maps before the linear layers.
        raise NotImplementedError("TODO: Implement ConvNet forward")

---
# Part 5: Load CIFAR-10 Dataset

In [ ]:
# Download and prepare CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

print("Loading CIFAR-10 dataset...")
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Use smaller batch size for from-scratch CPU implementation
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: {batch_size}")

---
# Part 6: Training Loop

Train your ConvNet on CIFAR-10 for a few epochs.

In [ ]:
# Initialize model and optimizer
model = ConvNet()
model.to(device)   # move params to the device BEFORE building the optimizer,
                   # so MomentumSGD snapshots velocity buffers on the right device
optimizer = MomentumSGD(model.parameters(), lr=0.01, momentum=0.9)
loss_fn = cross_entropy_loss

# Training configuration
num_epochs = 5
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print("Starting training...")
for epoch in range(num_epochs):
    start_time = time.time()
    
    # Train one epoch
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fn, device=device)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    
    # Evaluate on test set
    test_loss, test_acc = evaluate(model, test_loader, loss_fn, device=device)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1}/{num_epochs} ({elapsed:.1f}s) | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

print(f"\nTraining complete! Final test accuracy: {history['test_acc'][-1]:.4f}")

## Visualize Training Progress

In [ ]:
# Plot loss and accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['test_loss'], label='Test Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy curve
axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o')
axes[1].plot(history['test_acc'], label='Test Accuracy', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Classification Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"\nBest test accuracy: {max(history['test_acc']):.4f}")

## Sample Predictions

Visualize some predictions from your model.

In [ ]:
# Get some test samples
model.eval()
cifar10_classes = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# Get a batch
x_batch, y_batch = next(iter(test_loader))
with torch.no_grad():
    logits = model.forward(x_batch[:9].to(device))   # inputs to the model's device
    preds = logits.argmax(dim=1).cpu()               # back to cpu for plotting

# Visualize
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    # Unnormalize image
    img = x_batch[i].numpy().transpose(1, 2, 0)
    img = (img + 1) / 2  # Undo normalization
    img = np.clip(img, 0, 1)
    
    ax.imshow(img)
    true_label = cifar10_classes[y_batch[i]]
    pred_label = cifar10_classes[preds[i]]
    color = 'green' if preds[i] == y_batch[i] else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label}', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
# Part 7: Comparison with PyTorch

Implement the same ConvNet architecture using PyTorch's built-in modules and compare performance.

In [ ]:
class ConvNetPyTorch(nn.Module):
    """Same architecture as ConvNet but using PyTorch modules."""
    
    def __init__(self):
        super().__init__()
        # Build the same architecture as your from-scratch ConvNet
        # Use nn.Conv2d, nn.MaxPool2d, nn.ReLU, nn.Linear, nn.Flatten
        raise NotImplementedError("TODO: Build PyTorch version of ConvNet")
    
    def forward(self, x):
        # TODO: Implement forward pass
        raise NotImplementedError("TODO: Implement forward pass")

In [ ]:
# Train PyTorch version
torch.manual_seed(42)
model_pt = ConvNetPyTorch().to(device)
optimizer_pt = torch.optim.SGD(model_pt.parameters(), lr=0.01, momentum=0.9)
loss_fn_pt = nn.CrossEntropyLoss()

num_epochs_pt = 5
history_pt = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print("Training PyTorch version...")
for epoch in range(num_epochs_pt):
    # Train
    model_pt.train()
    train_loss, train_correct = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        logits = model_pt(x)
        loss = loss_fn_pt(logits, y)
        
        optimizer_pt.zero_grad()
        loss.backward()
        optimizer_pt.step()
        
        train_loss += loss.item() * x.shape[0]
        train_correct += (logits.argmax(dim=1) == y).sum().item()
    
    train_loss /= len(train_dataset)
    train_acc = train_correct / len(train_dataset)
    
    # Evaluate
    model_pt.eval()
    test_loss, test_correct = 0.0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            logits = model_pt(x)
            loss = loss_fn_pt(logits, y)
            test_loss += loss.item() * x.shape[0]
            test_correct += (logits.argmax(dim=1) == y).sum().item()
    
    test_loss /= len(test_dataset)
    test_acc = test_correct / len(test_dataset)
    
    history_pt['train_loss'].append(train_loss)
    history_pt['train_acc'].append(train_acc)
    history_pt['test_loss'].append(test_loss)
    history_pt['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs_pt} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

print(f"\nPyTorch training complete! Final test accuracy: {history_pt['test_acc'][-1]:.4f}")

## Comparison Plot

In [ ]:
# Compare accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# From-scratch vs PyTorch test accuracy
axes[0].plot(history['test_acc'], label='From-scratch', marker='o', linewidth=2)
axes[0].plot(history_pt['test_acc'], label='PyTorch', marker='s', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Test Accuracy Comparison')
axes[0].legend()
axes[0].grid(True)

# From-scratch vs PyTorch train loss
axes[1].plot(history['train_loss'], label='From-scratch', marker='o', linewidth=2)
axes[1].plot(history_pt['train_loss'], label='PyTorch', marker='s', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Train Loss')
axes[1].set_title('Train Loss Comparison')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"From-scratch test accuracy: {history['test_acc'][-1]:.4f}")
print(f"PyTorch test accuracy:      {history_pt['test_acc'][-1]:.4f}")
print(f"\nDifference: {abs(history['test_acc'][-1] - history_pt['test_acc'][-1]):.4f}")

# Submission Checklist

Before submitting, verify all code cells have been executed and outputs are visible.